# add-cxr-to-multimodal-task
---
## Setup

In [1]:
%%capture

from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import polars as pl

# PyHealth Packages
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ClinicalNotesMIMIC4, ClinicalNotesICDLabsMIMIC4, ClinicalNotesICDLabsCXRMIMIC4
from pyhealth.tasks.base_task import BaseTask

# Load MIMIC4 Files
# There's probably better ways dealing with this on the cluster, but working locally for now 
# (see: https://github.com/sunlabuiuc/PyHealth/blob/master/examples/mortality_prediction/multimodal_mimic4_minimal.py)

TASK = "ClinicalNotesICDLabsCXRMIMIC4" # The idea here is that we want additive tasks so we can evaluate the value in adding more modalities

PYHEALTH_REPO_ROOT = '/Users/wpang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimiciv/2.2")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimic-iv-note/2.2")
CXR_ROOT = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/physionet.org/files/mimic-cxr-jpg/2.0.0")
CACHE_DIR = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/wp/pyhealth_cache")

DEV_MODE = True

if __name__ == "__main__":

    if TASK == "ClinicalNotesMIMIC4": # A bit janky setup at the moment and open to iteration, but conveys the point for now
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )
    
    elif TASK == 'ClinicalNotesICDLabsMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

    elif TASK == 'ClinicalNotesICDLabsCXRMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                cxr_root=CXR_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cxr_tables=["metadata", "negbio"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

In [2]:
# negbio_patient_ids = (                                                                                                
#       dataset.global_event_df                                                                                           
#       .filter(pl.col("event_type") == "negbio")                                                                         
#       .select("patient_id")                                                                                             
#       .unique()                                                                                                         
#       .collect(engine="streaming")
#       .to_series()                                                                                                      
#       .to_list()                                                                                                        
#   )  

# OR 
# dataset.unique_patient_ids[2]

ID = "15092875"
print(f"Patient ID is {ID}")

Patient ID is 15092875


In [3]:
# Single patient
patient = dataset.get_patient(ID)  

### XRay

In [4]:
%%capture
negbio_events = patient.get_events(event_type="negbio")
print(len(negbio_events))
print(getattr(negbio_events[0], "pneumonia", None))
print(negbio_events[4])

In [5]:
%%capture
metadata_events = patient.get_events(event_type="metadata")
print(metadata_events)

### Run Task 

In [6]:
# Apply multimodal task
task = ClinicalNotesICDLabsCXRMIMIC4() 
samples = task(patient)

In [7]:
samples

[{'patient_id': '15092875',
  'discharge_note_times': ([" \nName:  ___                    Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   M\n \nService: MEDICINE\n \nAllergies: \nPatient recorded as having No Known Allergies to Drugs\n \nAttending: ___\n \nChief Complaint:\ndyspnea on exertion\n \nMajor Surgical or Invasive Procedure:\nnone\n\n \nHistory of Present Illness:\nHPI: This is a ___ year old male with chief complaint of DOE. \nThis delightful patient presented on ___ with a persistant \ncough that had been significantly worsening over three days. A \nchest xray at the ___ showed multifocal infiltrates c/w \natypical pneumonia. He was treated with 10 days of levaquin with \nresolution of his cough. However, he has become progressively \nweaker and developed severe DOE. On ___ he was seen for \nworsening DOE. His infiltrates on his cxr were worse. His 02 sat \nwas 93 ___nd 91 with exertion. He had new bilate